# Piping a RunnableParallel with Other Runnables

In [1]:
# Run the line of code below to check the version of langchain in the current environment.
# Substitute "langchain" with any other package name to check their version.

In [2]:
pip show langchain

Name: langchain
Version: 0.0.200
Summary: Building applications with LLMs through composability
Home-page: https://www.github.com/hwchase17/langchain
Author: 
Author-email: 
License: MIT
Location: /opt/anaconda3/envs/langchain_env/lib/python3.10/site-packages
Requires: aiohttp, async-timeout, dataclasses-json, langchainplus-sdk, numexpr, numpy, openapi-schema-pydantic, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
%load_ext dotenv
%dotenv

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [5]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [11]:
chat = ChatOpenAI(model_name = 'gpt-4o-mini', 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 500)

In [12]:
string_parser = StrOutputParser()

In [13]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [9]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [14]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests\n2. Developing a simple Machine Learning application with Scikit-learn\n3. Creating a GUI application with Tkinter or PyQt'}

In [15]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [16]:
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [17]:
print(chain_time2.invoke({'programming language':'Python'}))

The time it takes to complete the literature and projects can vary significantly based on your current skill level, the depth of understanding you wish to achieve, and the complexity of the projects. However, I can provide a rough estimate for each.

### Literature

1. **"Fluent Python" by Luciano Ramalho**  
   - **Estimated Time**: 4-6 weeks  
   - This book is comprehensive and covers advanced Python concepts. Depending on your pace and how much you practice the examples, it could take a month or more.

2. **"Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin**  
   - **Estimated Time**: 2-3 weeks  
   - This book is more concise and can be read relatively quickly, especially if you focus on the specific items that resonate with you.

3. **"Python Cookbook" by David Beazley and Brian K. Jones**  
   - **Estimated Time**: 3-4 weeks  
   - This book is practical and hands-on, so you might want to spend time working through the recipes, which could extend the t

In [18]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            